## US Presidential Elections (2000 - 2024): Data Processing

Sources:
- MIT Election Data and Science Lab. (2018). County Presidential Election Returns 2000-2024 (Version V20) [dataset]. Harvard Dataverse. https://doi.org/10.7910/DVN/VOQCHQ
- U.S. Census Bureau. (2000–2024). TIGER/Line shapefiles [Data set]. U.S. Department of Commerce. https://www.census.gov/geographies/mapping-files/time-series/geo/tiger-line-file.html

### Imports

In [1]:
import os
import geopandas as gpd
import pandas as pd

In [2]:
DATA_PATH = os.path.join("..", "..", "..", "data")
pres_election_df_raw = pd.read_csv(os.path.join(DATA_PATH, "raw", "us_county_pres_2000-2024.csv"))

In [3]:
pres_election_df_raw.head()

,state,county_name,year,state_po,county_fips,office,candidate,party,candidatevotes,totalvotes,version,mode
0,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,OTHER,OTHER,293.0,28281,20260225,TOTAL
1,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,CHASE OLIVER,LIBERTARIAN,65.0,28281,20260225,TOTAL
2,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,KAMALA D HARRIS,DEMOCRAT,7439.0,28281,20260225,TOTAL
3,ALABAMA,AUTAUGA,2024,AL,1001.0,US PRESIDENT,DONALD J TRUMP,REPUBLICAN,20484.0,28281,20260225,TOTAL
4,ALABAMA,BALDWIN,2024,AL,1003.0,US PRESIDENT,OTHER,OTHER,1276.0,122249,20260225,TOTAL


In [4]:
pres_election_df_raw.shape

(94151, 12)

### Preliminary Data Formatting

In [5]:
# Drop columns
pres_election_df = pres_election_df_raw.drop(columns=[
    "state_po",
    "office",
    "version"
])

# Convert to int
pres_election_df["county_fips"] = pres_election_df["county_fips"].astype("Int64")
pres_election_df["candidatevotes"] = pres_election_df["candidatevotes"].astype("Int64")

In [6]:
pres_election_df.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode
0,ALABAMA,AUTAUGA,2024,1001,OTHER,OTHER,293,28281,TOTAL
1,ALABAMA,AUTAUGA,2024,1001,CHASE OLIVER,LIBERTARIAN,65,28281,TOTAL
2,ALABAMA,AUTAUGA,2024,1001,KAMALA D HARRIS,DEMOCRAT,7439,28281,TOTAL
3,ALABAMA,AUTAUGA,2024,1001,DONALD J TRUMP,REPUBLICAN,20484,28281,TOTAL
4,ALABAMA,BALDWIN,2024,1003,OTHER,OTHER,1276,122249,TOTAL


There is an inconsistency in the FIPS for Kansas City, Missouri. We shall standardise its FIPS code to 2938000.

In [7]:
pres_election_df[pres_election_df["county_name"] == "KANSAS CITY"]

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode
9174,MISSOURI,KANSAS CITY,2024,36000,OTHER,OTHER,1498,124288,TOTAL
9175,MISSOURI,KANSAS CITY,2024,36000,KAMALA D HARRIS,DEMOCRAT,95660,124288,TOTAL
9176,MISSOURI,KANSAS CITY,2024,36000,CHASE OLIVER,LIBERTARIAN,905,124288,TOTAL
9177,MISSOURI,KANSAS CITY,2024,36000,DONALD J TRUMP,REPUBLICAN,26225,124288,TOTAL
26686,MISSOURI,KANSAS CITY,2000,2938000,AL GORE,DEMOCRAT,0,0,TOTAL
26687,MISSOURI,KANSAS CITY,2000,2938000,GEORGE W. BUSH,REPUBLICAN,0,0,TOTAL
26688,MISSOURI,KANSAS CITY,2000,2938000,OTHER,OTHER,0,0,TOTAL
26689,MISSOURI,KANSAS CITY,2000,2938000,RALPH NADER,GREEN,0,0,TOTAL
37771,MISSOURI,KANSAS CITY,2004,2938000,GEORGE W. BUSH,REPUBLICAN,36061,141423,TOTAL
37772,MISSOURI,KANSAS CITY,2004,2938000,JOHN KERRY,DEMOCRAT,104625,141423,TOTAL


In [8]:
KANSAS_CITY_MO_FIPS = 2938000

pres_election_df["county_fips"] = pres_election_df.apply(
    lambda row: KANSAS_CITY_MO_FIPS if ((row["state"] == "MISSOURI") and (row["county_name"] == "KANSAS CITY"))
    else row["county_fips"], axis=1
)

We format the county names of Alaska. Some rows have leading zeroes in the county (eg. District 01) while others do not (eg. District 1), so we make these consistent by removing all leading zeroes. We also remove District 99, given that it is used to categorise and track statewide absentee, military, and overseas ballots that are processed centrally rather than at a local precinct.

In [9]:
def format_ak_county_name(row):
    county_name = row["county_name"]
    
    if row["state"] != "ALASKA":
        return county_name
    
    return f"DISTRICT {int(county_name.split()[1])}"

pres_election_df["county_name"] = pres_election_df.apply(format_ak_county_name, axis=1)
pres_election_df = pres_election_df[
    (pres_election_df["state"] != "ALASKA") | (pres_election_df["county_name"] != "DISTRICT 99")
]

In [10]:
pres_election_df.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode
0,ALABAMA,AUTAUGA,2024,1001,OTHER,OTHER,293,28281,TOTAL
1,ALABAMA,AUTAUGA,2024,1001,CHASE OLIVER,LIBERTARIAN,65,28281,TOTAL
2,ALABAMA,AUTAUGA,2024,1001,KAMALA D HARRIS,DEMOCRAT,7439,28281,TOTAL
3,ALABAMA,AUTAUGA,2024,1001,DONALD J TRUMP,REPUBLICAN,20484,28281,TOTAL
4,ALABAMA,BALDWIN,2024,1003,OTHER,OTHER,1276,122249,TOTAL


In [11]:
pres_election_df.shape

(94126, 9)

Note that multiple rows may have the same `[state, county_name, year, county_fips, candidate, party, mode]`. This does not necessarily signal erroneous data. Later, after aggregating these rows, the combined vote counts tally with the corresponding `totalvotes` values.

In [12]:
duplicates = pres_election_df[pres_election_df.duplicated(
    ["state", "county_name", "year", "county_fips", "candidate", "party", "mode"], keep=False
)]
duplicates

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode
480,ARIZONA,COCONINO,2024,4005,UNDERVOTES,OTHER,452,70993,TOTAL
486,ARIZONA,COCONINO,2024,4005,UNDERVOTES,OTHER,0,70993,TOTAL
487,ARIZONA,COCONINO,2024,4005,UNDERVOTES,OTHER,0,70993,PROVISIONAL
491,ARIZONA,COCONINO,2024,4005,UNDERVOTES,OTHER,0,70993,ELECTION DAY
494,ARIZONA,COCONINO,2024,4005,UNDERVOTES,OTHER,0,70993,ELECTION DAY
...,...,...,...,...,...,...,...,...,...
16075,SOUTH CAROLINA,YORK,2024,45091,CHASE OLIVER,LIBERTARIAN,2,150059,FAILSAFE PROVISIONAL
16076,SOUTH CAROLINA,YORK,2024,45091,DONALD J TRUMP,REPUBLICAN,20,150059,FAILSAFE PROVISIONAL
16077,SOUTH CAROLINA,YORK,2024,45091,DONALD J TRUMP,REPUBLICAN,106,150059,FAILSAFE PROVISIONAL
16078,SOUTH CAROLINA,YORK,2024,45091,KAMALA D HARRIS,DEMOCRAT,12,150059,FAILSAFE PROVISIONAL


### Handling Missing Values

#### Missing Values in Mode

We only care about TOTAL mode for now - ignore the breakdown of election day / early votes / etc. Based on description of dataset, the default mode is TOTAL. So we can impute null mode values with TOTAL.

In [13]:
pres_election_df[pres_election_df["mode"].isna()].head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode
3770,IDAHO,ADA,2024,16001,OTHER,OTHER,5998,267419,NaN
3771,IDAHO,ADA,2024,16001,KAMALA D HARRIS,DEMOCRAT,116116,267419,NaN
3772,IDAHO,ADA,2024,16001,DONALD J TRUMP,REPUBLICAN,143759,267419,NaN
3773,IDAHO,ADA,2024,16001,CHASE OLIVER,LIBERTARIAN,1546,267419,NaN
3774,IDAHO,ADAMS,2024,16003,CHASE OLIVER,LIBERTARIAN,9,2681,NaN


The following confirms that no two rows have the same `[state, county_name, year, county_fips, candidate, party, mode]`, such that one row has TOTAL mode and one has null mode. Thus, imputing null mode with TOTAL mode does not lead to duplicated data.

In [14]:
key = ["state", "county_name", "year", "county_fips", "candidate", "party"]

nan_mode_rows = pres_election_df[pres_election_df["mode"].isna()]
total_mode_rows = pres_election_df[pres_election_df["mode"] == "TOTAL"]

overlap = nan_mode_rows.merge(
    total_mode_rows[key],
    on=key,
    how="inner"
)

overlap

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,mode


In [15]:
# Fill in null mode with TOTAL
pres_election_df["mode"] = pres_election_df["mode"].fillna("TOTAL")

In [16]:
# Only consider TOTAL mode
pres_election_df = pres_election_df[pres_election_df["mode"] == "TOTAL"]

In [17]:
pres_election_df.shape

(74682, 9)

In [18]:
pres_election_df.isna().sum()

state               0
county_name         0
year                0
county_fips        52
candidate           0
party             501
candidatevotes      4
totalvotes          0
mode                0
dtype: int64

In [19]:
pres_election_df = pres_election_df.drop(columns=["mode"])

#### Missing Values in Party

Rows with missing party correspond to total votes / undervotes / spoiled.

In [20]:
rows_with_missing_party = pres_election_df[pres_election_df["party"].isna()]
rows_with_missing_party.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
14829,SOUTH CAROLINA,ABBEVILLE,2024,45001,TOTAL VOTES CAST,NaN,12048,12048
14857,SOUTH CAROLINA,AIKEN,2024,45003,TOTAL VOTES CAST,NaN,86091,86091
14885,SOUTH CAROLINA,ALLENDALE,2024,45005,TOTAL VOTES CAST,NaN,3023,3023
14913,SOUTH CAROLINA,ANDERSON,2024,45007,TOTAL VOTES CAST,NaN,98296,98296
14941,SOUTH CAROLINA,BAMBERG,2024,45009,TOTAL VOTES CAST,NaN,5694,5694


In [21]:
rows_with_missing_party["candidate"].value_counts(dropna=False)

candidate
TOTAL VOTES CAST    427
UNDERVOTES           37
OVERVOTES            23
SPOILED              14
Name: count, dtype: int64

For all rows with candidate = TOTAL VOTES CAST, observe that `candidatevotes` and `totalvotes` correspond to the same value. On this note, we can remove these rows since we already have the `totalvotes` column.

In [22]:
rows_representing_total_votes = pres_election_df[pres_election_df["candidate"] == "TOTAL VOTES CAST"]
rows_representing_total_votes[
    rows_representing_total_votes["candidatevotes"]
    != rows_representing_total_votes["totalvotes"]
]

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes


In [23]:
# Remove candidate = TOTAL VOTES CAST
pres_election_df = pres_election_df[pres_election_df["candidate"] != "TOTAL VOTES CAST"]

In [24]:
pres_election_df.shape

(74255, 8)

Most of the SPOILED / UNDERVOTES / OVERVOTES are actually assigned to OTHER as the party value.

In [25]:
# Rows assigned to OTHER party
rows_representing_other_parties = pres_election_df[pres_election_df["party"] == "OTHER"]["candidate"]
rows_representing_other_parties.value_counts(dropna=False)

candidate
OTHER         20793
UNDERVOTES      116
OVERVOTES       114
Name: count, dtype: int64

In [26]:
# Rows representing SPOILED, UNDERVOTES, OVERVOTES
EXCEPTIONAL_VOTE_TYPES = {"SPOILED", "UNDERVOTES", "OVERVOTES"}
rows_representing_other_votes = pres_election_df[pres_election_df["candidate"].isin(EXCEPTIONAL_VOTE_TYPES)]
rows_representing_other_votes["party"].value_counts(dropna=False)

party
OTHER    230
NaN       74
Name: count, dtype: int64

We shall assign EXCEPTIONAL party to all cases of SPOILED, UNDERVOTES, OVERVOTES. This distinguishes them from actual OTHER candidates.

In [27]:
# Map to missing party value for all SPOILED, UNDERVOTES, OVERVOTES
EXCEPTIONAL_PARTY_NAME = "EXCEPTIONAL"
pres_election_df["party"] = pres_election_df.apply(
    lambda row: EXCEPTIONAL_PARTY_NAME if row["candidate"] in EXCEPTIONAL_VOTE_TYPES else row["party"],
    axis=1
)

In [28]:
pres_election_df.isna().sum()

state              0
county_name        0
year               0
county_fips       52
candidate          0
party              0
candidatevotes     4
totalvotes         0
dtype: int64

#### Missing Values in County FIPS

For rows with missing county FIPS, they do not correspond to results of a specific county (eg. statewide write-ins, UOCAVA in Maine, Federal Precinct in Rhode Island). For the purposes of the current project, we are concerned with county-level results only. We will thus remove these rows. In addition, the vote share of the rows removed is rather small and can be omitted for the time being.

In [29]:
pres_election_df[pres_election_df["county_fips"].isna()]

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
21802,CONNECTICUT,STATEWIDE WRITEIN,2000,<NA>,AL GORE,DEMOCRAT,0,0
21803,CONNECTICUT,STATEWIDE WRITEIN,2000,<NA>,GEORGE W. BUSH,REPUBLICAN,0,0
21804,CONNECTICUT,STATEWIDE WRITEIN,2000,<NA>,OTHER,OTHER,0,0
21805,CONNECTICUT,STATEWIDE WRITEIN,2000,<NA>,RALPH NADER,GREEN,0,0
25286,MAINE,MAINE UOCAVA,2000,<NA>,AL GORE,DEMOCRAT,0,0
25287,MAINE,MAINE UOCAVA,2000,<NA>,GEORGE W. BUSH,REPUBLICAN,0,0
25288,MAINE,MAINE UOCAVA,2000,<NA>,OTHER,OTHER,0,0
25289,MAINE,MAINE UOCAVA,2000,<NA>,RALPH NADER,GREEN,0,0
29802,RHODE ISLAND,FEDERAL PRECINCT,2000,<NA>,AL GORE,DEMOCRAT,0,0
29803,RHODE ISLAND,FEDERAL PRECINCT,2000,<NA>,GEORGE W. BUSH,REPUBLICAN,0,0


In [30]:
# Remove rows with empty county FIPS
pres_election_df = pres_election_df[~pres_election_df["county_fips"].isna()]

In [31]:
pres_election_df.shape

(74203, 8)

In [32]:
pres_election_df.isna().sum()

state             0
county_name       0
year              0
county_fips       0
candidate         0
party             0
candidatevotes    4
totalvotes        0
dtype: int64

#### Missing Values in Candidate Votes

There are 4 instances of missing candidate votes.

In [33]:
rows_with_missing_cand_votes = pres_election_df[pres_election_df["candidatevotes"].isna()]
rows_with_missing_cand_votes

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
93226,NEW MEXICO,DE BACA,2024,35011,CHASE OLIVER,LIBERTARIAN,<NA>,867
93287,NEW MEXICO,GUADALUPE,2024,35019,CHASE OLIVER,LIBERTARIAN,<NA>,1928
93296,NEW MEXICO,HARDING,2024,35021,CHASE OLIVER,LIBERTARIAN,<NA>,425
93326,NEW MEXICO,HIDALGO,2024,35023,CHASE OLIVER,LIBERTARIAN,<NA>,1861


We consider each of the counties affected for this particular year. Ignoring missing values, we sum up the votes of the other candidates and see whether the derived total count equals the total votes provided.

In [34]:
counties_with_rows_with_missing_cand_votes = pres_election_df[
    (pres_election_df["state"] == "NEW MEXICO")
    & (pres_election_df["year"] == 2024)
    & (pres_election_df["county_fips"].isin([35011, 35019, 35021, 35023]))
]

counties_with_rows_with_missing_cand_votes.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
93224,NEW MEXICO,DE BACA,2024,35011,OTHER,OTHER,12,867
93225,NEW MEXICO,DE BACA,2024,35011,KAMALA D HARRIS,DEMOCRAT,206,867
93226,NEW MEXICO,DE BACA,2024,35011,CHASE OLIVER,LIBERTARIAN,<NA>,867
93227,NEW MEXICO,DE BACA,2024,35011,DONALD J TRUMP,REPUBLICAN,649,867
93282,NEW MEXICO,GUADALUPE,2024,35019,KAMALA D HARRIS,DEMOCRAT,959,1928


In [35]:
grouped_votes_without_na_votes = counties_with_rows_with_missing_cand_votes \
    .groupby(["state", "county_name", "year", "county_fips"])["candidatevotes"] \
    .sum() \
    .reset_index()

grouped_votes_without_na_votes = grouped_votes_without_na_votes.rename(columns={"candidatevotes": "derived_total_votes"})

grouped_votes_without_na_votes

,state,county_name,year,county_fips,derived_total_votes
0,NEW MEXICO,DE BACA,2024,35011,867
1,NEW MEXICO,GUADALUPE,2024,35019,1928
2,NEW MEXICO,HARDING,2024,35021,425
3,NEW MEXICO,HIDALGO,2024,35023,1861


In [36]:
joined_df_with_derived_total_votes = counties_with_rows_with_missing_cand_votes.merge(
    grouped_votes_without_na_votes,
    on=["state", "county_name", "year", "county_fips"],
    how="inner"
)

joined_df_with_derived_total_votes

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,derived_total_votes
0,NEW MEXICO,DE BACA,2024,35011,OTHER,OTHER,12,867,867
1,NEW MEXICO,DE BACA,2024,35011,KAMALA D HARRIS,DEMOCRAT,206,867,867
2,NEW MEXICO,DE BACA,2024,35011,CHASE OLIVER,LIBERTARIAN,<NA>,867,867
3,NEW MEXICO,DE BACA,2024,35011,DONALD J TRUMP,REPUBLICAN,649,867,867
4,NEW MEXICO,GUADALUPE,2024,35019,KAMALA D HARRIS,DEMOCRAT,959,1928,1928
5,NEW MEXICO,GUADALUPE,2024,35019,DONALD J TRUMP,REPUBLICAN,945,1928,1928
6,NEW MEXICO,GUADALUPE,2024,35019,OTHER,OTHER,24,1928,1928
7,NEW MEXICO,GUADALUPE,2024,35019,CHASE OLIVER,LIBERTARIAN,<NA>,1928,1928
8,NEW MEXICO,HARDING,2024,35021,KAMALA D HARRIS,DEMOCRAT,128,425,425
9,NEW MEXICO,HARDING,2024,35021,CHASE OLIVER,LIBERTARIAN,<NA>,425,425


In [37]:
joined_df_with_derived_total_votes[
    joined_df_with_derived_total_votes["totalvotes"]
    != joined_df_with_derived_total_votes["derived_total_votes"]
]

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes,derived_total_votes


The derived total counts coincide with the total counts specified. Hence, we shall assume that Chase Oliver received 0 votes for each of these counties in 2024. While this might not be the case, it allows us to make use of the existing `totalvotes` column. In any case, the vote counts for these cases are likely to be insignificant. Subsequently, we impute the missing vote counts with zero values.

In [38]:
pres_election_df["candidatevotes"] = pres_election_df["candidatevotes"].fillna(0)

In [39]:
pres_election_df.shape

(74203, 8)

In [40]:
pres_election_df.isna().sum()

state             0
county_name       0
year              0
county_fips       0
candidate         0
party             0
candidatevotes    0
totalvotes        0
dtype: int64

#### Handling Inconsistencies

Generally, the total vote count includes exceptional votes, such as spoiled / overvotes / undervotes / etc. However, there are anomalies - DC and Wyoming in 2024.

In [41]:
agg_county_and_year_df = pres_election_df \
    .groupby(["state", "county_name", "year", "county_fips", "totalvotes"])["candidatevotes"] \
    .sum() \
    .reset_index()

inconsistent_df = agg_county_and_year_df[agg_county_and_year_df["candidatevotes"] != agg_county_and_year_df["totalvotes"]]
inconsistent_df

,state,county_name,year,county_fips,totalvotes,candidatevotes
2225,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,325879,328414
21062,WYOMING,ALBANY,2024,56001,17877,17993
21069,WYOMING,BIG HORN,2024,56003,5735,5788
21076,WYOMING,CAMPBELL,2024,56005,18342,18450
21083,WYOMING,CARBON,2024,56007,6387,6446
21090,WYOMING,CONVERSE,2024,56009,6739,6780
21097,WYOMING,CROOK,2024,56011,4344,4368
21104,WYOMING,FREMONT,2024,56013,17256,17396
21111,WYOMING,GOSHEN,2024,56015,6174,6214
21118,WYOMING,HOT SPRINGS,2024,56017,2620,2643


For DC in 2024, the undervotes and overvotes are excluded from the total vote count. To ensure consistency, we modify the total vote counts to include these ballots.

In [42]:
dc_wy_2024_df = pres_election_df[(pres_election_df["year"] == 2024) & (pres_election_df["state"].isin(["DISTRICT OF COLUMBIA", "WYOMING"]))]
dc_wy_2024_df

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
2845,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,UNDERVOTES,EXCEPTIONAL,2075,325879
2846,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,OTHER,OTHER,10618,325879
2847,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,OVERVOTES,EXCEPTIONAL,460,325879
2848,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,KAMALA D HARRIS,DEMOCRAT,294185,325879
2849,DISTRICT OF COLUMBIA,DISTRICT OF COLUMBIA,2024,11001,DONALD J TRUMP,REPUBLICAN,21076,325879
...,...,...,...,...,...,...,...,...
20497,WYOMING,WESTON,2024,56045,DONALD J TRUMP,REPUBLICAN,3069,3512
20498,WYOMING,WESTON,2024,56045,KAMALA D HARRIS,DEMOCRAT,378,3512
20499,WYOMING,WESTON,2024,56045,OTHER,OTHER,18,3512
20500,WYOMING,WESTON,2024,56045,OVERVOTES,EXCEPTIONAL,1,3512


In [43]:
# Verify the total votes recorded for DC + WY in 2024 do not include exceptional votes
dc_wy_2024_df_no_exception = dc_wy_2024_df[dc_wy_2024_df["party"] != EXCEPTIONAL_PARTY_NAME]
dc_wy_2024_df_no_exception_counts = dc_wy_2024_df_no_exception.groupby(["state", "county_name", "year", "county_fips"])["candidatevotes"] \
    .sum() \
    .reset_index()
dc_wy_2024_df_no_exception_counts = dc_wy_2024_df_no_exception_counts.rename(columns={"candidatevotes": "non_exception_votes"})

merged_df = inconsistent_df.merge(dc_wy_2024_df_no_exception_counts, on=["state", "county_name", "year", "county_fips"], how="inner")
(merged_df["totalvotes"] == merged_df["non_exception_votes"]).all()

np.True_

In [44]:
# Modify total vote count
actual_total_votes_dc_wy_2024 = dc_wy_2024_df.groupby(["state", "county_name", "year", "county_fips"])["candidatevotes"] \
    .sum() \
    .reset_index()
actual_total_votes_dc_wy_2024 = actual_total_votes_dc_wy_2024.rename(columns={"candidatevotes": "total_votes_with_exceptions"})

pres_election_df = pres_election_df.merge(
    actual_total_votes_dc_wy_2024,
    on=["state", "county_name", "year", "county_fips"],
    how="left"
)

pres_election_df["totalvotes"] = pres_election_df["total_votes_with_exceptions"].fillna(pres_election_df["totalvotes"])
pres_election_df = pres_election_df.drop(columns=["total_votes_with_exceptions"])

In [45]:
agg_county_and_year_df = pres_election_df \
    .groupby(["state", "county_name", "year", "county_fips", "totalvotes"])["candidatevotes"] \
    .sum() \
    .reset_index()

inconsistent_df = agg_county_and_year_df[agg_county_and_year_df["candidatevotes"] != agg_county_and_year_df["totalvotes"]]
inconsistent_df

,state,county_name,year,county_fips,totalvotes,candidatevotes


In [46]:
pres_election_df.shape

(74203, 8)

### Further Data Formatting

In [47]:
pres_election_df.head()

,state,county_name,year,county_fips,candidate,party,candidatevotes,totalvotes
0,ALABAMA,AUTAUGA,2024,1001,OTHER,OTHER,293,28281
1,ALABAMA,AUTAUGA,2024,1001,CHASE OLIVER,LIBERTARIAN,65,28281
2,ALABAMA,AUTAUGA,2024,1001,KAMALA D HARRIS,DEMOCRAT,7439,28281
3,ALABAMA,AUTAUGA,2024,1001,DONALD J TRUMP,REPUBLICAN,20484,28281
4,ALABAMA,BALDWIN,2024,1003,OTHER,OTHER,1276,122249


In [48]:
pres_election_df["party"].unique()

<StringArray>
['OTHER', 'LIBERTARIAN', 'DEMOCRAT', 'REPUBLICAN', 'EXCEPTIONAL', 'GREEN']
Length: 6, dtype: str

In [49]:
# Format parties
PARTY_NAME_MAPPING = {
    "DEMOCRAT": "DEMOCRATIC PARTY",
    "REPUBLICAN": "REPUBLICAN PARTY",
    "LIBERTARIAN": "LIBERTARIAN PARTY",
    "GREEN": "GREEN PARTY",
    "EXCEPTIONAL": "EXCEPTIONAL"
}

OTHER_PARTY_NAME = "OTHER"
pres_election_df["party"] = pres_election_df["party"].apply(
    lambda name: PARTY_NAME_MAPPING.get(name, OTHER_PARTY_NAME)
)

We only want to store party affiliation if the candidate name is known. As long as the candidate name is OTHER, we store it in the database as OTHER_CANDIDATE (taken to be a minor candidate). Hence, for these cases, we convert the party to OTHER as well.

In [50]:
# If candidate is OTHER, convert party to OTHER as well
OTHER_CANDIDATES_NAME = "OTHER"
pres_election_df["party"] = pres_election_df.apply(
    lambda row: OTHER_PARTY_NAME if row["candidate"] == OTHER_CANDIDATES_NAME else row["party"],
    axis=1
)

Likewise, we only want to store candidate names if the party is known. As long as the party name is OTHER, we store it in the database as OTHER CANDIDATE (taken to be a minor candidate). Hence, for these cases, we convert the candidate name to OTHER as well. 

In [51]:
# To account for multiple "OTHER" candidates (if the dataset has them),
# for each year and county, we convert the names of these minor candidates to a common name
# so that we can group them
pres_election_df["candidate"] = pres_election_df.apply(
    lambda row: OTHER_CANDIDATES_NAME if row["party"] == OTHER_PARTY_NAME else row["candidate"],
    axis=1
)

In [52]:
# Verify that all total vote values are the same for a given county in a given year
are_total_votes_consistent_across_county_and_year = pres_election_df \
    .groupby(["state", "county_name", "year", "county_fips"])["totalvotes"] \
    .nunique() == 1

are_total_votes_consistent_across_county_and_year.all()

np.True_

In [53]:
# Perform aggregation. This aggregates any rows corresponding to the same candidate, county and year
pres_election_df = pres_election_df \
    .groupby(["state", "county_name", "year", "county_fips", "candidate", "party", "totalvotes"])["candidatevotes"] \
    .sum() \
    .reset_index()

pres_election_df = pres_election_df[[
    "year",
    "state",
    "county_name",
    "county_fips",
    "candidate",
    "party",
    "candidatevotes",
    "totalvotes", 
]]

pres_election_df = pres_election_df.rename(
    columns={
        "candidatevotes": "votes",
        "totalvotes": "total_votes"
    }
)

In [54]:
# Verify that for each county in a given year, the sum of the candidate votes == the total votes
agg_county_and_year_df = pres_election_df \
    .groupby(["state", "county_name", "year", "county_fips", "total_votes"])["votes"] \
    .sum() \
    .reset_index()

(agg_county_and_year_df["votes"] == agg_county_and_year_df["total_votes"]).all()

np.True_

In [55]:
pres_election_df.head()

,year,state,county_name,county_fips,candidate,party,votes,total_votes
0,2000,ALABAMA,AUTAUGA,1001,AL GORE,DEMOCRATIC PARTY,4942,17208
1,2000,ALABAMA,AUTAUGA,1001,GEORGE W. BUSH,REPUBLICAN PARTY,11993,17208
2,2000,ALABAMA,AUTAUGA,1001,OTHER,OTHER,113,17208
3,2000,ALABAMA,AUTAUGA,1001,RALPH NADER,GREEN PARTY,160,17208
4,2004,ALABAMA,AUTAUGA,1001,GEORGE W. BUSH,REPUBLICAN PARTY,15196,20081


In [56]:
pres_election_df.shape

(71893, 8)

### Border Data

In [57]:
YEARS = [2000, 2008, 2012, 2016, 2020, 2024]

#### State Data

In [58]:
states_df_combined = None

for year in YEARS:
    states_df_for_year = gpd.read_file(os.path.join(DATA_PATH, "raw", "us_state_borders", f"tl_{year}_us_state.zip"))

    if year == 2000:
        states_df_for_year = states_df_for_year.rename(columns={"NAME00": "NAME"})

    states_df_for_year = states_df_for_year[["NAME", "geometry"]]

    states_df_for_year["NAME"] = states_df_for_year["NAME"].str.upper()
    states_df_for_year = states_df_for_year.to_crs(4326)  # convert geometry to compatible type

    states_df_for_year = states_df_for_year.rename(columns={"NAME": "state"})
    states_df_for_year["year"] = year
    states_df_for_year = states_df_for_year.sort_values(by=["state", "year"]).reset_index(drop=True)
    
    states_df_for_year = states_df_for_year[["state", "year", "geometry"]]

    if states_df_combined is None:
        states_df_combined = states_df_for_year
    else:
        states_df_combined = pd.concat([states_df_combined, states_df_for_year], axis=0, ignore_index=True)

In [59]:
states_df_combined.head()

,state,year,geometry
0,ALABAMA,2000,"POLYGON ((-85.51361 34.52382, -85.51304 34.521..."
1,ALASKA,2000,"MULTIPOLYGON (((177.44593 52.11134, 177.44302 ..."
2,ARIZONA,2000,"POLYGON ((-113.91599 36.99998, -113.91555 36.9..."
3,ARKANSAS,2000,"MULTIPOLYGON (((-92.11462 36.49804, -92.11463 ..."
4,CALIFORNIA,2000,"MULTIPOLYGON (((-119.00093 33.5359, -119.00093..."


The only places that do not have complete border data lie outside of the 50 states. That is fine as our project only pertains to the 50 states + DC for now.

In [60]:
required_years = set(YEARS)

# Years present for each state
years_by_state = states_df_combined.groupby("state")["year"].apply(set)

# States missing one or more required years
missing = {
    fips: sorted(required_years - years)
    for fips, years in years_by_state.items()
    if not required_years.issubset(years)
}

if missing:
    print("States with missing border data:")
    for fips, years in missing.items():
        print(f"{fips}: {years}")
else:
    print("No states with missing border data!")

States with missing border data:
AMERICAN SAMOA: [2000]
COMMONWEALTH OF THE NORTHERN MARIANA ISLANDS: [2000]
GUAM: [2000]
UNITED STATES VIRGIN ISLANDS: [2000, 2008]
VIRGIN ISLANDS OF THE UNITED STATES: [2000, 2012, 2016, 2020, 2024]


#### County Data

First, we handle counties from all states except Alaska. The Alaskan "counties" recorded in the county datasets are boroughs, which does not tally with the election data we have (the election data only includes results for each state house district). Hence, we need to download the borders for Alaskan state house districts separately.

In [61]:
non_ak_counties_df_combined = None

for year in YEARS:
    counties_df_for_year = gpd.read_file(os.path.join(DATA_PATH, "raw", "us_county_borders", f"tl_{year}_us_county.zip"))

    # Column names in 2000 and 2008 are formatted differently
    if year == 2000:
        counties_df_for_year = counties_df_for_year.rename(columns={"STATEFP00": "STATEFP", "CNTYIDFP00": "GEOID", "NAME00": "NAME"})
    elif year == 2008:
        counties_df_for_year = counties_df_for_year.rename(columns={"CNTYIDFP": "GEOID"})

    # Remove data for Alaska - we will add in state house district borders later
    counties_df_for_year = counties_df_for_year[counties_df_for_year["STATEFP"] != "02"]
    counties_df_for_year = counties_df_for_year.drop(columns=["STATEFP"])

    counties_df_for_year = counties_df_for_year[["GEOID", "NAME", "geometry"]]

    counties_df_for_year["GEOID"] = counties_df_for_year["GEOID"].astype("Int64")
    counties_df_for_year["NAME"] = counties_df_for_year["NAME"].str.upper()
    counties_df_for_year = counties_df_for_year.to_crs(4326)  # convert geometry to compatible type

    counties_df_for_year = counties_df_for_year.rename(columns={"GEOID": "county_fips", "NAME": "county_name"})
    
    counties_df_for_year["year"] = year
    counties_df_for_year = counties_df_for_year[["county_fips", "year", "geometry"]]

    if non_ak_counties_df_combined is None:
        non_ak_counties_df_combined = counties_df_for_year
    else:
        non_ak_counties_df_combined = pd.concat([non_ak_counties_df_combined, counties_df_for_year], axis=0, ignore_index=True)

In [63]:
non_ak_counties_df_combined.head()

,county_fips,year,geometry
0,28101,2000,"POLYGON ((-89.13497 32.57697, -89.13466 32.576..."
1,28027,2000,"POLYGON ((-90.59062 33.9869, -90.59473 33.9869..."
2,22065,2000,"MULTIPOLYGON (((-91.03511 32.12035, -91.03621 ..."
3,51003,2000,"POLYGON ((-78.76043 37.91433, -78.76247 37.917..."
4,51540,2000,"POLYGON ((-78.45448 38.05383, -78.45428 38.053..."


Here, we process border data for Alaska's state house districts.

In [64]:
ak_districts_df_combined = None

for year in YEARS:
    ak_districts_df_for_year = gpd.read_file(os.path.join(DATA_PATH, "raw", "alaska_state_house_district_borders", f"tl_{year}_02_sldl.zip"))

    # Column names in 2000 and 2008 are formatted differently
    if year == 2000:
        ak_districts_df_for_year = ak_districts_df_for_year.rename(columns={"NAMELSAD00": "NAMELSAD"})

        # In 2000, there was no GEOID - make it manually
        ak_districts_df_for_year["GEOID"] = ak_districts_df_for_year["STATEFP00"] + ak_districts_df_for_year["SLDLST00"].str.zfill(3)
        
    elif year == 2008:
        ak_districts_df_for_year = ak_districts_df_for_year.rename(columns={"SLDLIDFP": "GEOID"})

    ak_districts_df_for_year = ak_districts_df_for_year[["GEOID", "NAMELSAD", "geometry"]]

    ak_districts_df_for_year["GEOID"] = ak_districts_df_for_year["GEOID"].astype("Int64")
    ak_districts_df_for_year["NAMELSAD"] = ak_districts_df_for_year["NAMELSAD"].str.upper()
    ak_districts_df_for_year = ak_districts_df_for_year.to_crs(4326)  # convert geometry to compatible type

    ak_districts_df_for_year = ak_districts_df_for_year.rename(columns={"GEOID": "county_fips", "NAMELSAD": "county_name"})

    ak_districts_df_for_year["year"] = year
    ak_districts_df_for_year = ak_districts_df_for_year[["county_fips", "year", "geometry"]]

    if ak_districts_df_combined is None:
        ak_districts_df_combined = ak_districts_df_for_year
    else:
        ak_districts_df_combined = pd.concat([ak_districts_df_combined, ak_districts_df_for_year], axis=0, ignore_index=True)

In [65]:
ak_districts_df_combined.head()

,county_fips,year,geometry
0,2040,2000,"MULTIPOLYGON (((-157.2016 58.84494, -157.19964..."
1,2036,2000,"POLYGON ((-152.19035 62.95815, -152.16579 62.9..."
2,2009,2000,"POLYGON ((-151.10744 60.49338, -151.10745 60.4..."
3,2008,2000,"POLYGON ((-151.10812 60.49304, -151.10745 60.4..."
4,2007,2000,"POLYGON ((-153.26472 60.64087, -153.25972 60.6..."


We also need to get border data for Kansas City, Missouri, because that is in our election data.

In [66]:
kansas_city_df_combined = None

for year in YEARS:
    mo_places_df_for_year = gpd.read_file(os.path.join(DATA_PATH, "raw", "missouri_place_borders", f"tl_{year}_29_place.zip"))

    # Column names in 2000 and 2008 are formatted differently
    if year == 2000:
        mo_places_df_for_year = mo_places_df_for_year.rename(columns={"NAME00": "NAME", "PLCIDFP00": "GEOID"})
    elif year == 2008:
        mo_places_df_for_year = mo_places_df_for_year.rename(columns={"PLCIDFP": "GEOID"})

    mo_places_df_for_year = mo_places_df_for_year[["GEOID", "NAME", "geometry"]]
    mo_places_df_for_year["GEOID"] = mo_places_df_for_year["GEOID"].astype("Int64")

    kansas_city_df_for_year = mo_places_df_for_year[mo_places_df_for_year["GEOID"] == KANSAS_CITY_MO_FIPS]

    kansas_city_df_for_year["NAME"] = kansas_city_df_for_year["NAME"].str.upper()
    kansas_city_df_for_year = kansas_city_df_for_year.to_crs(4326)  # convert geometry to compatible type

    kansas_city_df_for_year = kansas_city_df_for_year.rename(columns={"GEOID": "county_fips", "NAME": "county_name"})

    kansas_city_df_for_year["year"] = year
    kansas_city_df_for_year = kansas_city_df_for_year[["county_fips", "year", "geometry"]]

    if kansas_city_df_combined is None:
        kansas_city_df_combined = kansas_city_df_for_year
    else:
        kansas_city_df_combined = pd.concat([kansas_city_df_combined, kansas_city_df_for_year], axis=0, ignore_index=True)

In [67]:
kansas_city_df_combined.head()

,county_fips,year,geometry
0,2938000,2000,"MULTIPOLYGON (((-94.76558 39.28834, -94.76516 ..."
1,2938000,2008,"POLYGON ((-94.44113 38.83993, -94.4414 38.8399..."
2,2938000,2012,"POLYGON ((-94.76558 39.28834, -94.76516 39.290..."
3,2938000,2016,"POLYGON ((-94.76592 39.28707, -94.76558 39.288..."
4,2938000,2020,"POLYGON ((-94.7655 39.28988, -94.76544 39.2903..."


In [ ]:
# Add the Alaska and Kansas City, MO borders to the other county borders
counties_df_combined = pd.concat([non_ak_counties_df_combined, ak_districts_df_combined, kansas_city_df_combined], axis=0, ignore_index=True)
counties_df_combined = counties_df_combined.sort_values(by=["county_fips", "year"]).reset_index(drop=True)

In [69]:
counties_df_combined.head()

,county_fips,year,geometry
0,1001,2000,"POLYGON ((-86.62619 32.70638, -86.62498 32.706..."
1,1001,2008,"POLYGON ((-86.41245 32.57084, -86.41244 32.569..."
2,1001,2012,"POLYGON ((-86.9212 32.65754, -86.92093 32.6579..."
3,1001,2016,"POLYGON ((-86.9031 32.54063, -86.90313 32.5410..."
4,1001,2020,"POLYGON ((-86.9031 32.54063, -86.90312 32.5408..."


There are counties which do not have complete border data. But these are to be expected:

- Broomfield County, CO (08014) - Created in 2001, so no 2000 geometry.
- Connecticut old counties (09001–09015) - Replaced by 9 planning regions as county-equivalent units in 2022, so no 2024 geometry.
- Connecticut planning regions (09110–09190) - New county-equivalent units, so no geometry for elections before 2022.
- Oglala Lakota County, SD (46102) - Created/renamed from Shannon County in 2015, so no earlier geometry.
- Shannon County, SD (46113) - Renamed to Oglala Lakota County, so no geometry from 2016 onward.
- Bedford City, VA (51515) - Consolidated with Bedford County in 2013, so no geometry from 2016 onward.
- Clifton Forge, VA (51560) - Consolidated into Alleghany County in 2001, so no geometry from 2008 onward.
- 600xx, 660xx, 690xx, 780xx - U.S. territories/island areas; excluded because our project only concerns the 50 states + DC for now.

In [70]:
required_years = set(YEARS)

# Check missing years
years_by_county = counties_df_combined.groupby("county_fips")["year"].apply(set)

missing = {
    fips: sorted(required_years - years)
    for fips, years in years_by_county.items()
    if not required_years.issubset(years)
}

# Check duplicate county-year rows
duplicates = counties_df_combined[counties_df_combined.duplicated(["county_fips", "year"], keep=False)]

if missing:
    print("Counties with missing border data:")
    for fips, years in missing.items():
        print(f"{fips}: {years}")
else:
    print("No counties with missing border data!")

Counties with missing border data:
8014: [2000]
9001: [2024]
9003: [2024]
9005: [2024]
9007: [2024]
9009: [2024]
9011: [2024]
9013: [2024]
9015: [2024]
9110: [2000, 2008, 2012, 2016, 2020]
9120: [2000, 2008, 2012, 2016, 2020]
9130: [2000, 2008, 2012, 2016, 2020]
9140: [2000, 2008, 2012, 2016, 2020]
9150: [2000, 2008, 2012, 2016, 2020]
9160: [2000, 2008, 2012, 2016, 2020]
9170: [2000, 2008, 2012, 2016, 2020]
9180: [2000, 2008, 2012, 2016, 2020]
9190: [2000, 2008, 2012, 2016, 2020]
46102: [2000, 2008, 2012]
46113: [2016, 2020, 2024]
51515: [2016, 2020, 2024]
51560: [2008, 2012, 2016, 2020, 2024]
60010: [2000]
60020: [2000]
60030: [2000]
60040: [2000]
60050: [2000]
66010: [2000]
69085: [2000]
69100: [2000]
69110: [2000]
69120: [2000]
78010: [2000]
78020: [2000]
78030: [2000]


### Save to CSV

In [71]:
os.makedirs(os.path.join(DATA_PATH, "processed"), exist_ok=True)
pres_election_df.to_csv(os.path.join(DATA_PATH, "processed", "us_county_pres_2000-2024_processed.csv"), index=False)
states_df_combined.to_csv(os.path.join(DATA_PATH, "processed", "us_state_borders_2000-2024_processed.csv"), index=False)
counties_df_combined.to_csv(os.path.join(DATA_PATH, "processed", "us_county_borders_2000-2024_processed.csv"), index=False)